# 01 数据理解与清洗

**项目**：Udacity 免费试听筛选实验 A/B 评估
**本 Notebook 范围**：实验背景与漏斗 → 字段口径核对 → 数据质量体检 → 14 天观察窗截断处理（双窗口）→ 长/宽格式分析表 → 核心指标构建 → 落盘 processed。
**边界**：本 Notebook 只做描述性统计与清洗，**不做任何假设检验与效果推断**。

- 原始数据：`data/raw/Final_Project_Results_{Control,Experiment}.csv`（只读，37 天日粒度聚合）
- 产出位置：`data/processed/`
- 可复现：从上到下顺序执行即可（Python 3.13 + pandas 2.3.2）。

## 实验背景与转化漏斗

- **业务改动**：用户点击 *Start Free Trial* 后，实验组新增“每周可投入学习时间”筛选：≥5 小时/周者照常进入注册结账；<5 小时/周者看到“课程通常需要更多时间投入”的提示，可改为使用免费课程材料。对照组无此步骤。
- **分流单位（unit of diversion）**：cookie；用户一旦 enroll，后续以 user-id 跟踪；未 enroll 用户没有 user-id。
- **漏斗**：`Pageviews（看到课程概览页的 unique cookies）→ Clicks（点击 Start Free Trial 的 unique cookies）→ Enrollments（完成注册进入 14 天免费试用）→ Payments（14 天后仍留存并产生付费）`。
- **Payments 的日期口径**：该列日期是 **enroll 的起始日**，而非实际扣款日；需要 enroll 后满 14 天才能观察到是否 Payment，因此实验后段日期的 Enrollments/Payments 尚未成熟。
- **业务假设**：筛选会减少低意愿 enroll（Gross Conversion 下降），但不显著减少最终付费（Net Conversion 不下降）。

In [1]:
# 加载与全局设置
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

ctrl = pd.read_csv(RAW / "Final_Project_Results_Control.csv")
exp = pd.read_csv(RAW / "Final_Project_Results_Experiment.csv")
print("raw shapes:", ctrl.shape, exp.shape)
ctrl.head(3)

raw shapes: (37, 5) (37, 5)


,Date,Pageviews,Clicks,Enrollments,Payments
0,"Sat, Oct 11",7723,687,134.0,70.0
1,"Sun, Oct 12",9102,779,147.0,70.0
2,"Mon, Oct 13",10511,909,167.0,95.0


## 字段口径核对

| 字段 | 口径（与 Udacity 课程定义一致） |
|---|---|
| Date | 日期（原始无年份，按星期前缀核验为 2014 年） |
| Pageviews | 当日看到课程概览页的 unique cookies 数（≈ Cookies，分流规模） |
| Clicks | 当日点击 Start Free Trial 的 unique cookies 数（筛选步骤之前发生） |
| Enrollments | 当日进入免费试用注册的 user-id 数 |
| Payments | 当日 enroll、且 14 天后仍留存付费的 user-id 数 |

In [2]:
# 解析日期（年份 2014 已在后续分析 用星期前缀与年历一致性验证）+ 组标签
def prep(df: pd.DataFrame, group: str) -> pd.DataFrame:
    df = df.copy()
    md = df["Date"].str.split(",").str[1].str.strip()          # "Oct 11"
    df["Date"] = pd.to_datetime("2014 " + md, format="%Y %b %d")
    df["Group"] = group
    df["Weekday"] = df["Date"].dt.day_name().str[:3]
    return df

ctrl = prep(ctrl, "Control")
exp = prep(exp, "Experiment")
daily = pd.concat([ctrl, exp], ignore_index=True)
print(daily.dtypes)
daily.head(3)

Date           datetime64[ns]
Pageviews               int64
Clicks                  int64
Enrollments           float64
Payments              float64
Group                  object
Weekday                object
dtype: object


,Date,Pageviews,Clicks,Enrollments,Payments,Group,Weekday
0,2014-10-11,7723,687,134.0,70.0,Control,Sat
1,2014-10-12,9102,779,147.0,70.0,Control,Sun
2,2014-10-13,10511,909,167.0,95.0,Control,Mon


## 数据质量体检

检查：数据类型、日期连续性、重复、缺失、负值、漏斗单调（PV≥Clicks≥Enrollments≥Payments）、极端波动日（组内 z 分数，仅标记不删除）。

In [3]:
# 8.1 结构：行数、重复、日期连续性、缺失、负值、漏斗单调性
qc = {}
qc["rows_per_group"] = daily.groupby("Group").size().to_dict()
qc["duplicated_date_group"] = int(daily.duplicated(["Date", "Group"]).sum())

for g, d in [("Control", ctrl), ("Experiment", exp)]:
    gaps = d["Date"].sort_values().diff().dropna().dt.days
    assert (gaps == 1).all(), f"{g} 日期不连续"
    assert len(d) == (d["Date"].max() - d["Date"].min()).days + 1
qc["date_range"] = {g: [str(d["Date"].min().date()), str(d["Date"].max().date()), len(d)]
                    for g, d in [("Control", ctrl), ("Experiment", exp)]}
qc["missing"] = {k: int(v) for k, v in daily.isna().sum().items()}
qc["negatives"] = {c_: int((daily[c_] < 0).sum())
                   for c_ in ["Pageviews", "Clicks", "Enrollments", "Payments"]}
od = daily[daily["Enrollments"].notna()]
qc["funnel_violations"] = {
    "Pageviews_lt_Clicks": int((od["Pageviews"] < od["Clicks"]).sum()),
    "Clicks_lt_Enrollments": int((od["Clicks"] < od["Enrollments"]).sum()),
    "Enrollments_lt_Payments": int((od["Enrollments"] < od["Payments"]).sum()),
}
for k, v in qc.items():
    print(k, "->", v)

rows_per_group -> {'Control': 37, 'Experiment': 37}
duplicated_date_group -> 0
date_range -> {'Control': ['2014-10-11', '2014-11-16', 37], 'Experiment': ['2014-10-11', '2014-11-16', 37]}
missing -> {'Date': 0, 'Pageviews': 0, 'Clicks': 0, 'Enrollments': 28, 'Payments': 28, 'Group': 0, 'Weekday': 0}
negatives -> {'Pageviews': 0, 'Clicks': 0, 'Enrollments': 0, 'Payments': 0}
funnel_violations -> {'Pageviews_lt_Clicks': 0, 'Clicks_lt_Enrollments': 0, 'Enrollments_lt_Payments': 0}


In [4]:
# 8.2 极端波动日：组内 |z|>=2.5 仅标记、不删除（删除没有业务依据）
flags = []
for g, d in [("Control", ctrl.copy()), ("Experiment", exp.copy())]:
    d["CTP"] = d["Clicks"] / d["Pageviews"]
    for col in ["Pageviews", "Clicks", "CTP"]:
        z = (d[col] - d[col].mean()) / d[col].std(ddof=1)
        for idx in d.index[z.abs() >= 2.5]:
            flags.append({"Group": g, "Date": d.loc[idx, "Date"].date().isoformat(),
                          "Metric": col, "Value": float(d.loc[idx, col]),
                          "z": float(z.loc[idx])})
flags_df = pd.DataFrame(flags)
print(flags_df.to_string(index=False))
print("\n说明：10/18 为周六低流量（周末季节性，两组同形态）；10/24 两组 CTP 同向下探，属共同外部因素，保留并在后续分析分流质量检查中复核。")

     Group       Date    Metric       Value         z
   Control 2014-10-18 Pageviews 7434.000000 -2.573491
   Control 2014-10-24       CTP    0.071338 -3.342337
Experiment 2014-10-24       CTP    0.074133 -2.585822

说明：10/18 为周六低流量（周末季节性，两组同形态）；10/24 两组 CTP 同向下探，属共同外部因素，保留并在后续分析分流质量检查中复核。


## 14 天观察窗截断 → 双窗口口径

- **流量窗（全 37 天，2014-10-11 → 11-16）**：Pageviews/Clicks 在所有日期都成熟 → 用于 SRM 与 invariant metrics。
- **outcome 窗（前 23 天，2014-10-11 → 11-02）**：Enrollments/Payments 仅这 23 天完整可观察 → Gross/Net/PayPerEnrollment 等转化分析只用该窗，分母 Clicks 也只取这 23 天。
- 被排除：11-03 → 11-16 共 14 天；下面量化样本损失。

In [5]:
# 9.1 窗口标记 + 样本损失量化
daily["OutcomeComplete"] = daily["Enrollments"].notna() & daily["Payments"].notna()
loss_rows = []
for g, d in daily.groupby("Group"):
    full_clicks = int(d["Clicks"].sum())
    out_clicks = int(d.loc[d["OutcomeComplete"], "Clicks"].sum())
    loss_rows.append({"Group": g, "Clicks_37d": full_clicks, "Clicks_outcome_23d": out_clicks,
                      "Clicks_excluded": full_clicks - out_clicks,
                      "Pct_excluded": (full_clicks - out_clicks) / full_clicks,
                      "Outcome_days": int(d["OutcomeComplete"].sum()),
                      "Excluded_days": int((~d["OutcomeComplete"]).sum())})
window_loss = pd.DataFrame(loss_rows)
print(window_loss.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
excluded_dates = sorted(daily.loc[~daily["OutcomeComplete"], "Date"].dt.date.astype(str).unique())
print("被排除日期:", excluded_dates)

     Group  Clicks_37d  Clicks_outcome_23d  Clicks_excluded  Pct_excluded  Outcome_days  Excluded_days
   Control       28378               17293            11085        0.3906            23             14
Experiment       28325               17260            11065        0.3906            23             14
被排除日期: ['2014-11-03', '2014-11-04', '2014-11-05', '2014-11-06', '2014-11-07', '2014-11-08', '2014-11-09', '2014-11-10', '2014-11-11', '2014-11-12', '2014-11-13', '2014-11-14', '2014-11-15', '2014-11-16']


## 长格式 / 宽格式分析表

In [6]:
# 10.1 日粒度指标
daily["CTP"] = daily["Clicks"] / daily["Pageviews"]
daily["GrossConversion"] = daily["Enrollments"] / daily["Clicks"]
daily["NetConversion"] = daily["Payments"] / daily["Clicks"]
daily["PayPerEnrollment"] = daily["Payments"] / daily["Enrollments"]

daily_long = daily[["Date", "Weekday", "Group", "Pageviews", "Clicks", "Enrollments",
                    "Payments", "OutcomeComplete", "CTP", "GrossConversion",
                    "NetConversion", "PayPerEnrollment"]].sort_values(
                    ["Date", "Group"]).reset_index(drop=True)

# 10.2 宽格式：同一日期两组并排
piv = daily.pivot(index="Date", columns="Group")
wide = piv.swaplevel(axis=1).sort_index(axis=1, level=0)
wide.columns = [f"{g}_{m}" for g, m in wide.columns]
wide = wide.reset_index()

daily_long.to_csv(PROC / "daily_long.csv", index=False, encoding="utf-8")
wide.to_csv(PROC / "daily_wide.csv", index=False, encoding="utf-8")
window_loss.to_csv(PROC / "window_sample_loss.csv", index=False, encoding="utf-8")
print("daily_long:", daily_long.shape, "| daily_wide:", wide.shape)
daily_long.head(4)

daily_long: (74, 12) | daily_wide: (37, 21)


,Date,Weekday,Group,Pageviews,Clicks,Enrollments,Payments,OutcomeComplete,CTP,GrossConversion,NetConversion,PayPerEnrollment
0,2014-10-11,Sat,Control,7723,687,134.0,70.0,True,0.088955,0.195051,0.101892,0.522388
1,2014-10-11,Sat,Experiment,7716,686,105.0,34.0,True,0.088906,0.153061,0.049563,0.323810
2,2014-10-12,Sun,Control,9102,779,147.0,70.0,True,0.085586,0.188703,0.089859,0.476190
3,2014-10-12,Sun,Experiment,9288,785,116.0,91.0,True,0.084518,0.147771,0.115924,0.784483


## 核心指标体系（pooled，描述性）

- **CTP** = Clicks / Pageviews（不变指标，全 37 天）
- **Gross Conversion** = Enrollments / Clicks（策略效果指标，23 天 outcome 窗）
- **Net Conversion** = Payments / Clicks（核心决策指标，23 天 outcome 窗）
- **Payments/Enrollments** = Payments / Enrollments（试听用户质量辅助指标，23 天）
> 此处只给点估计做数据理解；双样本 Z 检验、CI、Bootstrap、Delta Method 在后续分析。

In [7]:
# 11.1 双窗口 pooled 汇总
rows = []
for g, d in daily.groupby("Group"):
    ow = d[d["OutcomeComplete"]]
    rows.append({
        "Group": g,
        "Pageviews_37d": int(d["Pageviews"].sum()),
        "Clicks_37d": int(d["Clicks"].sum()),
        "CTP_37d": d["Clicks"].sum() / d["Pageviews"].sum(),
        "Outcome_days": int(ow.shape[0]),
        "Clicks_23d": int(ow["Clicks"].sum()),
        "Enrollments_23d": int(ow["Enrollments"].sum()),
        "Payments_23d": int(ow["Payments"].sum()),
        "GrossConversion": ow["Enrollments"].sum() / ow["Clicks"].sum(),
        "NetConversion": ow["Payments"].sum() / ow["Clicks"].sum(),
        "PayPerEnrollment": ow["Payments"].sum() / ow["Enrollments"].sum(),
    })
pooled = pd.DataFrame(rows)
pooled.to_csv(PROC / "pooled_summary.csv", index=False, encoding="utf-8")
print(pooled.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

# 11.2 描述性差异（实验-对照；非检验）
p = pooled.set_index("Group")
for m in ["CTP_37d", "GrossConversion", "NetConversion", "PayPerEnrollment"]:
    diff = p.loc["Experiment", m] - p.loc["Control", m]
    print(f"delta {m:18s} = {diff:+.6f}  ({diff/p.loc['Control',m]:+.2%} rel.)")

     Group  Pageviews_37d  Clicks_37d  CTP_37d  Outcome_days  Clicks_23d  Enrollments_23d  Payments_23d  GrossConversion  NetConversion  PayPerEnrollment
   Control         345543       28378 0.082126            23       17293             3785          2033         0.218875       0.117562          0.537120
Experiment         344660       28325 0.082182            23       17260             3423          1945         0.198320       0.112688          0.568215
delta CTP_37d            = +0.000057  (+0.07% rel.)
delta GrossConversion    = -0.020555  (-9.39% rel.)
delta NetConversion      = -0.004874  (-4.15% rel.)
delta PayPerEnrollment   = +0.031095  (+5.79% rel.)


In [8]:
# 落盘后回读验证（独立路径复核）
back_long = pd.read_csv(PROC / "daily_long.csv", parse_dates=["Date"])
back_wide = pd.read_csv(PROC / "daily_wide.csv", parse_dates=["Date"])
back_pool = pd.read_csv(PROC / "pooled_summary.csv")
assert len(back_long) == 74 and back_long["Group"].value_counts().to_dict() == {"Control": 37, "Experiment": 37}
assert len(back_wide) == 37
assert int(back_pool.set_index("Group").loc["Control", "Pageviews_37d"]) == 345543
assert int(back_pool.set_index("Group").loc["Experiment", "Payments_23d"]) == 1945
print("回读验证通过：daily_long 74 行、daily_wide 37 行、pooled 总量与数据核验一致。")
print("processed 文件：", sorted(p.name for p in PROC.glob("*.csv")))

回读验证通过：daily_long 74 行、daily_wide 37 行、pooled 总量与数据核验一致。
processed 文件： ['daily_long.csv', 'daily_wide.csv', 'pooled_summary.csv', 'window_sample_loss.csv']


## 小结（描述性，不做推断）

1. 数据无重复、无缺失型错误、无负值、漏斗单调全部成立；日期连续 37 天；仅后 14 天 outcome 未成熟。
2. 双窗口：流量分析用 37 天；转化分析用前 23 天（两组各约 39.06% 的点击落在被截断区间，无法用于转化分析）。
3. 描述性结果：Gross Conversion 对照 0.218875 / 实验 0.198320（−2.06pp）；Net Conversion 0.117562 / 0.112688（−0.49pp）；Pay/Enroll 0.537120 / 0.568215（+3.11pp）；CTP 两组几乎相同（0.08213 / 0.08218）。是否显著、是否非劣效，留待后续分析。